# Space–time cubes and cohort histories

**Atlas source reading copy · browser-network**

[Course guide](../index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Sea-Level-Rise/blob/66065673b47f9bc5c0f6e5009ac1c924a9f5200c/content/notebooks/08_space_time_cube.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/notebooks/08_space_time_cube.ipynb` at `66065673b47f9bc5c0f6e5009ac1c924a9f5200c`; original SHA-256 `30de885d1030840dc482edb1b2658dcd0f6d12d01eccbfcbc14eadfd1ffff3af`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# Space–time cubes and cohort histories

**Time:** 35 minutes · **Prerequisites:** basic Python; run cells from top to bottom.

Construct a space–time cube from public aggregates; trace a city history; compare a cube with a heatmap.

[Run this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/08_space_time_cube.ipynb) · [Full-screen dashboard](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/dashboard/) · [Official poster](https://plus.iasociety.org/sites/default/files/2024-09/e-poster_755.pdf)

Each notebook runs independently. Use **Run → Run All Cells**; the first package download can take a minute. Save a downloaded copy to keep your work. The read-only book includes executed examples.

In [ ]:
# First run downloads Python packages in JupyterLite; subsequent runs use browser caches.
import sys
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'plotly', 'ipywidgets'])
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
from coastlab import (DATA, SCENARIOS, YEARS, COLORS, load_data, select, cards,
                      toy_slr, toy_probability, cumulative_probability,
                      haversine_km, show, dashboard, land_traces)
facilities, exposure = load_data()
published = pd.read_csv(DATA / 'poster_aggregates.csv')
print('Ready: public poster aggregates + clearly labeled fictional facility fixtures.')

## Stack maps, not populations

Imagine each decade's map as a sheet of paper and stack the sheets in time order. Longitude and latitude locate a city; vertical position locates a year. Colour shows the published potentially flooded percentage. Repeated points are observations of the same reporting unit under different modeled years; they are not additional facilities.

Rotation can reveal patterns but also hide points. Offer a flat alternative with exact values. Keep the colour scale fixed across RCPs.

In [ ]:
city=published.query("scope == 'city' and scenario == 'RCP 8.5'")
fig=go.Figure()
for name,g in city.groupby('city'):
    fig.add_trace(go.Scatter3d(x=g.longitude,y=g.latitude,z=g.year,mode='lines+markers',name=name,
        marker=dict(size=7,color=g.percent,cmin=0,cmax=100,colorscale='YlOrRd'),
        text=[f'{name}: {n} / {d}' for n,d in zip(g.potentially_flooded,g.facilities_2023)],
        hovertemplate='%{text}<br>Year %{z}<extra></extra>'))
fig.update_layout(title='Public city time histories — vertical axis = year',scene=dict(xaxis_title='Longitude',yaxis_title='Latitude',zaxis_title='Year'))
show(fig)

In [ ]:
matrix=city.pivot(index='city',columns='year',values='percent')
fig=px.imshow(matrix,zmin=0,zmax=100,text_auto='.1f',aspect='auto',color_continuous_scale='YlOrRd',
    labels={'color':'Potentially flooded (%)'},title='The same cube unfolded into a heatmap')
show(fig)

## Lab: first crossing is not a predicted event date

Find the first sampled decade in which each city reaches 50% potentially flooded under each RCP. Return “not reached in sampled years” when appropriate. This is a threshold crossing in a modeled series, not the date of a real flood, and the ten-year sampling limits temporal precision.

In [ ]:
city_rows=published.query("scope == 'city'")
crossings=city_rows[city_rows.percent>=50].groupby(['scenario','city']).year.min()
all_keys=pd.MultiIndex.from_product([SCENARIOS,sorted(city_rows.city.unique())],names=['scenario','city'])
display(crossings.reindex(all_keys).rename('first_sampled_year_at_50_percent').to_frame())

**Worked check:** Bangkok's city row is already above 50% in 2030. Abidjan's city row first exceeds 50% in 2070. A missing value means the threshold is not reached in the available series; it should not become zero or 2020.

**Deliverable:** The cube, a heatmap and a one-sentence comparison of the questions each answers best. Avoid interpolating an exact transition date unless you state and defend an interpolation model.